### Backends in Deep Agents

In [ ]:
import os
import uuid
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import Literal
from tavily import TavilyClient
from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
from deepagents.backends import FilesystemBackend

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

In [2]:
# Model Initialization
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=groq_api_key)

### StateBackend

In [ ]:
# Create a Deep Agent (Both are same - one is implicit & the other one is explicitly declared)
agent1 = create_deep_agent(
    model=llm,
    system_prompt="Act as a researcher"
)

agent2 = create_deep_agent(
    model=llm,
    system_prompt="Act as a researcher",
    backend=StateBackend()  
)

In [ ]:
result = agent2.invoke({
    "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. upload video\n"
                "Then tell me you've done it."
            )
        }]
})

print("\n--------------- Agent Reply -------------------------\n")
print(result["messages"][-1].content)

---------------Agent Reply-------------------------

I've created **/notes/todo.txt** with the requested content.


In [6]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. Record video\n2. Edit video\n3. upload video\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='ee749be5-ab1c-423d-bf42-d9b9e1de4211'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to create file /notes/todo.txt with given content. Steps: ensure directory exists? Not sure if /notes exists. We can write file directly; write_file will create file, but if directory missing maybe error? Usually we can assume path exists. Use write_file tool. Then respond confirming.\n\n', 'tool_calls': [{'id': 'fc_ab10fbb0-d032-4d7e-b87e-be856273bbfd', 'function': {'arguments': '{"content":"1. Record video\\n2. Edit video\\n3. upload video","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 5551, 'total_tokens': 5665, 'completion_time': 0.237296982

In [ ]:
# Check if the backend is working
print("\n--------------- Agent Reply -------------------------\n")
files = result.get("files", {})

if files:
    print(f"StateBackend is working - {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\nPath: {path}\n{'-' * 50}\n{content}")
else:
    print("No files found in state. Either the agent didn't write a file,"
    "or the backend isn't wired up correctly.")


---------------Agent Reply-------------------------

StateBackend is working - 1 file(s) in state:

Path: /notes/todo.txt
--------------------------------------------------
{'content': '1. Record video\n2. Edit video\n3. upload video', 'encoding': 'utf-8', 'created_at': '2026-07-26T03:04:11.612978+00:00', 'modified_at': '2026-07-26T03:04:11.612978+00:00'}


In [ ]:
# Prove persistence within the same thread
followup = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me."
    }],
    "files": result.get("files", {})  # <--pass the virtual filesystem along
})

print("\n-------- Read back (same thread) ---------------")
print(followup["messages"][-1].content)


--------Read back (same thread)---------------
1. Record video  
2. Edit video  
3. upload video


### FilesystemBackend (local disk)

In [ ]:
# Invoke the agent and ask it to write a file
ROOT = "."

agent = create_deep_agent(
    model=llm,
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

print(f"Agent created with FilesystemBackend(root_dir={ROOT!r}).")
print("Files written by the agent will appear on your ACTUAL disk.")

Agent created with FilesystemBackend(root_dir='.').
Files written by the agent will appear on your ACTUAL disk.


In [ ]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

print("\n--------------------- Agent reply -----------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
Done.


In [ ]:
# Check the backend is working — look on the real disk
# With virtual_mode=True, /notes/todo.txt maps to ./notes/todo.txt
print("\n--- Backend check (real filesystem) -----------------------------")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print(f"FilesystemBackend is working — file exists on disk:")
    print(f"{disk_path.resolve()}\n{'-' * 40}")
    print(disk_path.read_text())
else:
    print(f"Expected file not found at {disk_path.resolve()}")
    print("The agent may not have called the write tool, or the path "
          "mapping differs.")


--- Backend check (real filesystem) -----------------------------
FilesystemBackend is working — file exists on disk:
C:\Users\viren\generative-ai-implementations\langchain\langchain-updated\deep-agents\notes\todo.txt
----------------------------------------
1. Record video
2. Edit video
3. Upload video


In [13]:
# Prove persistence ACROSS sessions:
# Unlike StateBackend, this file survives even after Python exits.
# A brand-new agent (fresh state) can read it back from disk.
fresh_agent = create_deep_agent(
    model=llm,
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me."
    }]
    # NOTE: no `files` state passed in — the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) ------")
print(followup["messages"][-1].content)


--- Read-back with a FRESH agent (proves disk persistence) ------
1. Record video  
2. Edit video  
3. Upload video


### Deep Agent — StoreBackend

In [ ]:
store = InMemoryStore()

agent = create_deep_agent(
    model=llm,
    backend=StoreBackend(
        # Local dev: static namespace. No deployment runtime needed.
        # In a LangSmith Deployment you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)

print("Agent created with StoreBackend + static namespace.")

Agent created with StoreBackend + static namespace.


In [ ]:
import os


# THREAD 1 — write a file
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you've done it."
            )
        }]
    },
    config=thread_1,
)

print("\n--- Agent reply (thread 1) --------------------------------------")
print(result["messages"][-1].content)


--- Agent reply (thread 1) --------------------------------------
Done.


In [ ]:
# Thread 2: read back on a DIFFERENT thread 
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me."
    }]},
    config=thread_2,
)
print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)


--- Read-back on a different thread ---
1. Record video  
2. Edit video  
3. Upload video
